In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
# Project root
PROJECT_DIR = Path.cwd().parent

# Raw data folder
DATA_DIR = PROJECT_DIR / "data" / "raw"

print(DATA_DIR)

f:\insurance_ai_copilot\data\raw


In [3]:
customers_df = pd.read_csv(DATA_DIR / "customers.csv")
policies_df = pd.read_csv(DATA_DIR / "policies.csv")
premiums_df = pd.read_csv(DATA_DIR / "premiums.csv")
claims_df = pd.read_csv(DATA_DIR / "claims.csv")
underwriting_df = pd.read_csv(DATA_DIR / "underwriting.csv")
renewals_df = pd.read_csv(DATA_DIR / "renewals.csv")
agents_df = pd.read_csv(DATA_DIR / "agents.csv")

print("All datasets loaded.")

All datasets loaded.


In [4]:
print("Customers:", customers_df.duplicated().sum())
print("Policies:", policies_df.duplicated().sum())
print("Premiums:", premiums_df.duplicated().sum())
print("Claims:", claims_df.duplicated().sum())
print("Underwriting:", underwriting_df.duplicated().sum())
print("Renewals:", renewals_df.duplicated().sum())
print("Agents:", agents_df.duplicated().sum())

Customers: 0
Policies: 0
Premiums: 0
Claims: 0
Underwriting: 0
Renewals: 0
Agents: 0


In [5]:
primary_keys = {
    "customers": (customers_df, "CUSTOMER_ID"),
    "policies": (policies_df, "POLICY_ID"),
    "premiums": (premiums_df, "PREMIUM_TXN_ID"),
    "claims": (claims_df, "CLAIM_ID"),
    "underwriting": (underwriting_df, "UNDERWRITING_ID"),
    "renewals": (renewals_df, "RENEWAL_ID"),
    "agents": (agents_df, "AGENT_ID")
}

In [6]:
for table_name, (df, key) in primary_keys.items():

    duplicate_keys = df[key].duplicated().sum()
    missing_keys = df[key].isna().sum()

    print(
        table_name,
        "| Duplicate IDs:", duplicate_keys,
        "| Missing IDs:", missing_keys
    )

customers | Duplicate IDs: 0 | Missing IDs: 0
policies | Duplicate IDs: 0 | Missing IDs: 0
premiums | Duplicate IDs: 0 | Missing IDs: 0
claims | Duplicate IDs: 0 | Missing IDs: 0
underwriting | Duplicate IDs: 0 | Missing IDs: 0
renewals | Duplicate IDs: 0 | Missing IDs: 0
agents | Duplicate IDs: 0 | Missing IDs: 0


In [7]:
invalid_customer = policies_df[
    ~policies_df["CUSTOMER_ID"].isin(
        customers_df["CUSTOMER_ID"]
    )
]

print("Invalid customer references:", len(invalid_customer))

Invalid customer references: 0


In [8]:
invalid_agent = policies_df[
    ~policies_df["AGENT_ID"].isin(
        agents_df["AGENT_ID"]
    )
]

print("Invalid agent references:", len(invalid_agent))

Invalid agent references: 0


In [9]:
invalid_claim_policy = claims_df[
    ~claims_df["POLICY_ID"].isin(
        policies_df["POLICY_ID"]
    )
]

print("Claims with invalid policy:", len(invalid_claim_policy))

Claims with invalid policy: 0


In [10]:
invalid_premium_policy = premiums_df[
    ~premiums_df["POLICY_ID"].isin(
        policies_df["POLICY_ID"]
    )
]

print("Premiums with invalid policy:", len(invalid_premium_policy))

Premiums with invalid policy: 0


In [11]:
claim_customer_check = claims_df.merge(
    policies_df[
        ["POLICY_ID", "CUSTOMER_ID"]
    ],
    on="POLICY_ID",
    how="left",
    suffixes=("_CLAIM", "_POLICY")
)

In [12]:
claim_customer_check[
    claim_customer_check["CUSTOMER_ID_CLAIM"]
    !=
    claim_customer_check["CUSTOMER_ID_POLICY"]
].shape[0]

0

In [13]:
for name, df in {
    "customers": customers_df,
    "policies": policies_df,
    "premiums": premiums_df,
    "claims": claims_df,
    "underwriting": underwriting_df,
    "renewals": renewals_df,
    "agents": agents_df
}.items():

    missing = df.isna().sum()

    missing = missing[missing > 0]

    print("\n", name.upper())
    print(missing)


 CUSTOMERS
Series([], dtype: int64)

 POLICIES
Series([], dtype: int64)

 PREMIUMS
PAID_DATE             8839
PAYMENT_DELAY_DAYS    8839
dtype: int64

 CLAIMS
Series([], dtype: int64)

 UNDERWRITING
PREMIUM_LOADING_PCT    1537
dtype: int64

 RENEWALS
Series([], dtype: int64)

 AGENTS
Series([], dtype: int64)


In [14]:
# Understand Why Premium Values Are Missing
premiums_df.groupby("PAYMENT_STATUS")["PAID_DATE"].apply(lambda x: x.isna().sum())

PAYMENT_STATUS
Failed     2014
Overdue    3979
Paid          0
Pending    2846
Name: PAID_DATE, dtype: int64

In [15]:
# Underwriting Missing Valuesunderwriting_df.groupby(
underwriting_df.groupby(
    "DECISION"
)["PREMIUM_LOADING_PCT"].apply(
    lambda x: x.isna().sum()
)

DECISION
Approved                    0
Approved with Loading       0
Declined                 1537
Name: PREMIUM_LOADING_PCT, dtype: int64

In [16]:
customers_clean = customers_df.copy()
policies_clean = policies_df.copy()
premiums_clean = premiums_df.copy()
claims_clean = claims_df.copy()
renewals_clean = renewals_df.copy()
agents_clean = agents_df.copy()
underwriting_clean = underwriting_df.copy()

In [17]:
customers_clean["CUSTOMER_SINCE"] = pd.to_datetime(
    customers_clean["CUSTOMER_SINCE"],
    errors="coerce"
)

In [18]:
policies_clean["POLICY_START_DATE"] = pd.to_datetime(
    policies_clean["POLICY_START_DATE"],
    errors="coerce"
)

policies_clean["POLICY_END_DATE"] = pd.to_datetime(
    policies_clean["POLICY_END_DATE"],
    errors="coerce"
)

In [19]:
claims_clean["INCIDENT_DATE"] = pd.to_datetime(
    claims_clean["INCIDENT_DATE"],
    errors="coerce"
)

claims_clean["REPORTED_DATE"] = pd.to_datetime(
    claims_clean["REPORTED_DATE"],
    errors="coerce"
)

In [20]:
renewals_clean["RENEWAL_DUE_DATE"] = pd.to_datetime(
    renewals_clean["RENEWAL_DUE_DATE"],
    errors="coerce"
)

In [21]:
invalid_policy_dates = policies_clean[
    policies_clean["POLICY_START_DATE"]
    >
    policies_clean["POLICY_END_DATE"]
]

len(invalid_policy_dates)

0

INCIDENT_DATE <= REPORTED_DATE

In [22]:
invalid_claim_dates = claims_clean[
    claims_clean["INCIDENT_DATE"]
    >
    claims_clean["REPORTED_DATE"]
]

len(invalid_claim_dates)

0

Major Insurance Business Rule

A claim incident would normally need to occur while the policy provides coverage.

In [23]:
claim_policy_check = claims_clean.merge(
    policies_clean[
        [
            "POLICY_ID",
            "POLICY_START_DATE",
            "POLICY_END_DATE"
        ]
    ],
    on="POLICY_ID",
    how="left"
)

Create flag:

In [24]:
claim_policy_check["CLAIM_OUTSIDE_POLICY"] = (
    (claim_policy_check["INCIDENT_DATE"]< claim_policy_check["POLICY_START_DATE"])
    |
    (claim_policy_check["INCIDENT_DATE"]> claim_policy_check["POLICY_END_DATE"])
)

In [25]:
claim_policy_check[
    "CLAIM_OUTSIDE_POLICY"
].value_counts()

CLAIM_OUTSIDE_POLICY
True     53985
False    21015
Name: count, dtype: int64

First document:

“A high percentage of claim incident dates do not fall within the associated policy start/end dates. This requires clarification with the source/business team before exclusion.”

Because maybe:

policy table contains latest policy period

while claims refer to older versions.

Or maybe:

synthetic data generator created inconsistent dates

We don't know until business confirms it.

Premium Due Date vs Policy Coverage

In [26]:
premium_policy_check = premiums_clean.merge(
    policies_clean[
        [
            "POLICY_ID",
            "POLICY_START_DATE",
            "POLICY_END_DATE"
        ]
    ],
    on="POLICY_ID",
    how="left"
)

In [27]:
premium_policy_check["DUE_DATE_OUTSIDE_POLICY"] = (
    (premium_policy_check["DUE_DATE"]< premium_policy_check["POLICY_START_DATE"])
    |
    (premium_policy_check["DUE_DATE"]> premium_policy_check["POLICY_END_DATE"])
)

In [28]:
premium_policy_check[
    "DUE_DATE_OUTSIDE_POLICY"
].value_counts()

DUE_DATE_OUTSIDE_POLICY
True     68045
False    31955
Name: count, dtype: int64

Claim Settlement Validation         

Settlement Amount <= Claim Amount

In [29]:
invalid_settlement = claims_clean[
    claims_clean["SETTLEMENT_AMOUNT"]>claims_clean["CLAIM_AMOUNT"]
]

len(invalid_settlement)

0

Premium Payment Logic

In [30]:
paid_transactions = premiums_clean[
    premiums_clean["PAYMENT_STATUS"] == "Paid"
]

paid_transactions[
    "PAID_DATE"
].isna().sum()

np.int64(0)

In [31]:
non_paid = premiums_clean[
    premiums_clean["PAYMENT_STATUS"] != "Paid"
]

non_paid[
    "PAID_DATE"
].notna().sum()

np.int64(0)

means payment status and payment date are internally consistent.

Verify Payment Delay

In [32]:
premiums_clean["PAID_DATE"] = pd.to_datetime(
    premiums_clean["PAID_DATE"],
    errors="coerce"
)

premiums_clean["DUE_DATE"] = pd.to_datetime(
    premiums_clean["DUE_DATE"],
    errors="coerce"
)

In [33]:
paid_transactions = premiums_clean[
    premiums_clean["PAYMENT_STATUS"] == "Paid"
].copy()

paid_transactions["CALCULATED_DELAY"] = (
    paid_transactions["PAID_DATE"]
    -
    paid_transactions["DUE_DATE"]
).dt.days

In [34]:
(paid_transactions["CALCULATED_DELAY"]!=paid_transactions["PAYMENT_DELAY_DAYS"]).sum()

np.int64(0)

Underwriting Logic

DECISION = Approved

we expect:

PREMIUM_LOADING_PCT = 0

In [35]:
underwriting_clean[(underwriting_clean["DECISION"] == "Approved")
    &
    (underwriting_clean["PREMIUM_LOADING_PCT"] > 0)
].shape[0]

0

In [36]:
underwriting_clean[
    (underwriting_clean["DECISION"]
     == "Approved with Loading")
    &
    (underwriting_clean["PREMIUM_LOADING_PCT"] <= 0)
].shape[0]

0

Check Numeric Ranges

In [37]:
customers_clean[
    [
        "AGE",
        "ANNUAL_INCOME",
        "CREDIT_SCORE"
    ]
].describe()

,AGE,ANNUAL_INCOME,CREDIT_SCORE
count,100000.000000,1.000000e+05,100000.000000
mean,38.671880,8.311229e+05,719.502090
std,11.513542,6.567305e+05,69.892151
min,18.000000,1.200000e+05,450.000000
25%,30.000000,4.065565e+05,672.000000
50%,38.000000,6.493635e+05,720.000000
75%,47.000000,1.044275e+06,767.000000
max,80.000000,8.000000e+06,900.000000


In [38]:
policies_clean[
    [
        "SUM_INSURED",
        "ANNUAL_PREMIUM",
        "RISK_SCORE",
        "DISCOUNT_PCT",
        "COMMISSION_PCT"
    ]
].describe()

,SUM_INSURED,ANNUAL_PREMIUM,RISK_SCORE,DISCOUNT_PCT,COMMISSION_PCT
count,1.500000e+05,150000.000000,150000.000000,150000.000000,150000.000000
mean,2.558792e+06,19195.141140,415.727507,10.007650,2.509821
std,2.960261e+06,14113.463906,68.643447,5.781533,1.440787
min,1.000000e+05,1200.000000,250.000000,0.000000,0.000000
25%,5.866960e+05,10513.000000,369.000000,5.010000,1.270000
50%,1.169048e+06,15628.000000,415.000000,10.010000,2.510000
75%,3.732162e+06,23199.000000,462.000000,15.020000,3.750000
max,2.500000e+07,186032.000000,744.000000,20.000000,5.000000


In [39]:
claims_clean[
    [
        "CLAIM_AMOUNT",
        "SETTLEMENT_AMOUNT",
        "PROCESSING_DAYS",
        "FRAUD_SCORE"
    ]
].describe()

,CLAIM_AMOUNT,SETTLEMENT_AMOUNT,PROCESSING_DAYS,FRAUD_SCORE
count,7.500000e+04,7.500000e+04,75000.000000,75000.000000
mean,3.063485e+05,1.967341e+05,20.512053,0.498369
std,6.084179e+05,4.773754e+05,12.152669,0.288381
min,2.000000e+03,0.000000e+00,1.000000,0.000000
25%,4.103175e+04,0.000000e+00,12.000000,0.247900
50%,1.090645e+05,4.814900e+04,18.000000,0.498500
75%,3.112128e+05,1.822520e+05,27.000000,0.748200
max,1.599195e+07,1.444330e+07,120.000000,1.000000


Check Categorical Values

In [40]:
customers_clean[
    "CUSTOMER_RISK_SEGMENT"
].value_counts(dropna=False)

CUSTOMER_RISK_SEGMENT
Low       54974
Medium    34012
High      11014
Name: count, dtype: int64

In [41]:
policies_clean[
    "POLICY_STATUS"
].value_counts()

POLICY_STATUS
Active       104959
Expired       21019
Lapsed        14914
Cancelled      9108
Name: count, dtype: int64

In [42]:
claims_clean[
    "FRAUD_FLAG"
].value_counts()

FRAUD_FLAG
No           72841
Suspected     1815
Confirmed      344
Name: count, dtype: int64

In [43]:
renewals_clean[
    "RENEWAL_STATUS"
].value_counts()

RENEWAL_STATUS
Renewed      17334
Lapsed        1463
Pending        796
Cancelled      407
Name: count, dtype: int64

Renewal Formula Validation

Premium Increase %

       New Premium - Old Premium
= ------------------------------- × 100
           Old Premium

In [44]:
renewals_clean["CALCULATED_INCREASE_PCT"] = (
    (
        renewals_clean["NEW_PREMIUM"]
        -
        renewals_clean["OLD_PREMIUM"]
    )
    /
    renewals_clean["OLD_PREMIUM"]
) * 100

In [45]:
renewals_clean[
    [
        "OLD_PREMIUM",
        "NEW_PREMIUM",
        "PREMIUM_INCREASE_PCT",
        "CALCULATED_INCREASE_PCT"
    ]
].head()

,OLD_PREMIUM,NEW_PREMIUM,PREMIUM_INCREASE_PCT,CALCULATED_INCREASE_PCT
0,11390,12441,9.23,9.227392
1,13513,13983,3.48,3.478132
2,20696,20942,1.19,1.188635
3,11112,11584,4.25,4.247660
4,13833,15043,8.75,8.747199


Create a Data Quality Summary

In [46]:
dq_summary = pd.DataFrame({
    "CHECK": [
        "Duplicate customer IDs",
        "Duplicate policy IDs",
        "Invalid policy customers",
        "Invalid policy agents",
        "Invalid claim policies",
        "Policy start after end",
        "Claim reported before incident",
        "Claims outside policy period",
        "Premium due outside policy period",
        "Settlement greater than claim"
    ],

    "ISSUE_COUNT": [
        customers_clean["CUSTOMER_ID"].duplicated().sum(),

        policies_clean["POLICY_ID"].duplicated().sum(),

        len(invalid_customer),

        len(invalid_agent),

        len(invalid_claim_policy),

        len(invalid_policy_dates),

        len(invalid_claim_dates),

        claim_policy_check[
            "CLAIM_OUTSIDE_POLICY"
        ].sum(),

        premium_policy_check[
            "DUE_DATE_OUTSIDE_POLICY"
        ].sum(),

        len(invalid_settlement)
    ]
})

dq_summary

,CHECK,ISSUE_COUNT
0,Duplicate customer IDs,0
1,Duplicate policy IDs,0
2,Invalid policy customers,0
3,Invalid policy agents,0
4,Invalid claim policies,0
5,Policy start after end,0
6,Claim reported before incident,0
7,Claims outside policy period,53985
8,Premium due outside policy period,68045
9,Settlement greater than claim,0


What did we learn?

The data has very strong technical integrity:

Primary keys → clean
Foreign keys → clean
Duplicates → clean
Customer mappings → clean
Payment calculations → clean
Underwriting logic → clean
Claim settlement logic → clean

But it has important business-rule inconsistencies:

~72% claims outside policy coverage

~68% premium due dates outside policy coverage

That distinction matters.

A dataset can be technically clean but business-wise inconsistent.